# Tutorial 12 — Bayesian Optimization for Reaction Conditions

**Goal.** Treat a reaction-condition space (catalyst/ligand/base/solvent/temperature/...) as a black
box and use Bayesian optimization (BO) with a Gaussian Process (GP) surrogate to find high-yield
conditions in as few "experiments" as possible — mirroring how BO is used in real automated/flow
chemistry campaigns (Perera et al. *Science* 2018; Reizman & Jensen *React. Chem. Eng.* 2016; Shields
et al. *Nature* 2021; Felton, Rittig & Lapkin *Chem. Methods* 2021, "Summit"). See
`PAPERS-AND-DATASETS.md` in this folder for the full literature summary and dataset provenance.

**How "experiments" work in this notebook.** We do **not** run real reactions. Each of the four
datasets in `data/` is a fixed table of (conditions → measured yield) pairs — three are real
experimental data mirrored from the original source repositories, one (Perera) is a clearly-labeled
synthetic-but-representative stand-in (see Section 1). "Running an experiment" means looking up the
yield of the matching row in that table. This is exactly the *closed-loop simulation against a fixed
historical dataset* methodology that Perera, Reizman, and especially Summit use to make BO strategies
comparable without consuming additional wet-lab time.

**Notebook structure** (numbers match the tutorial webpage's Instructions list):

1. Present the dataset(s) and variables
2. Featurize the reaction (precomputed)
3. Choose an initialization technique
4. Choose an acquisition function and its hyperparameters
5. Batch size (fixed at 4)
6. Preview the trajectory on a 2D plot before committing
7. Submit the design choices (checkpoint)
8. Launch the full run (5 seeds) and submit to the leaderboard

**Audience note.** This assumes chemistry background but not necessarily an ML one — equations shown
(GP posterior mean/variance, EI/UCB) are for intuition, not derived from scratch. Examples span
Pd-catalyzed cross-coupling reactions relevant to drug-discovery, process, and materials chemists.


## 0 · Setup

We use **BoTorch**/**GPyTorch** for the GP surrogate and acquisition functions (same stack as
`08-molecule-generation` and `11-doe`), plus RDKit for descriptor featurization and scikit-learn for
k-means / PCA / random projection.


In [ ]:
!pip -q install botorch gpytorch rdkit scikit-learn pandas matplotlib requests > /dev/null

import json, math, time, uuid, warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

import torch
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.kernels import MaternKernel, ScaleKernel
from botorch.acquisition import ExpectedImprovement, UpperConfidenceBound, qExpectedImprovement
from botorch.optim import optimize_acqf

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.random_projection import GaussianRandomProjection

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
RDLogger.DisableLog("rdApp.*")

warnings.filterwarnings("ignore", category=UserWarning)
torch.set_default_dtype(torch.float64)

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    # Standalone run (e.g. Colab without a full repo clone) -- fetch the 4 CSVs from GitHub raw.
    import urllib.request
    DATA_DIR.mkdir(exist_ok=True)
    base = "https://raw.githubusercontent.com/julschleinitz/ai4chemistry-bootcamp/main/tutorials/12-reaction-bo/data/"
    for fname in ["perera_suzuki.csv", "reizman_suzuki.csv", "shields_direct_arylation.csv", "baumgartner_cn_coupling.csv"]:
        try:
            urllib.request.urlretrieve(base + fname, DATA_DIR / fname)
        except Exception as e:
            print(f"Could not fetch {fname}: {e}")

GLOBAL_SEED = 0
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
print("Setup OK. Device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1 · Present the dataset(s) and variables

Four datasets are bundled in `data/`, each a fixed lookup table of (reaction conditions, yield).
Provenance and factor structure for each (see `PAPERS-AND-DATASETS.md` for full citations):

| key | paper | real/synthetic | reactions | factors |
|---|---|---|---|---|
| `perera_suzuki` | Perera et al., *Science* 2018 | **synthetic-representative** (see note below) | 5760 | electrophile (7) x ligand (12) x base (8) x solvent (4), + replicate |
| `reizman_suzuki` | Reizman & Jensen, *React. Chem. Eng.* 2016 | real (mirrored from `summit`) | ~96 (4 cases) | catalyst/ligand (cat.), residence time, temperature, catalyst loading |
| `shields_direct_arylation` | Shields et al., *Nature* 2021 | real (mirrored from `edbo`) | ~1536 (≈395 measured) | base, ligand, solvent (cat., as SMILES) x concentration x temperature |
| `baumgartner_cn_coupling` | Baumgartner et al. 2019, aggregated by Felton/Rittig/Lapkin ("Summit") | real (mirrored from `summit`) | 96 | catalyst (3), base (4), base equivalents, temperature, residence time |

**Why `perera_suzuki` is synthetic.** The full 5760-row Perera grid is not mirrored as a flat CSV in
any of the open repos we could reach in this environment (it lives behind the paper's SI tables). We
built a synthetic-but-representative stand-in with the *same* categorical factor structure (7
electrophiles x 12 ligands x 8 bases x 4 solvents, 2 replicates) and yields drawn from a smooth
function of the one-hot factor encoding plus noise, calibrated so the sample mean/std/max
approximately match the paper's reported summary statistics. It is explicitly labeled as such in its
CSV header and should not be used for anything beyond exercising this notebook's BO machinery -- see
`PAPERS-AND-DATASETS.md` for the authoritative source.

Set `DATASET` below to pick which one to work with for the rest of the notebook.


In [ ]:
DATASET = "perera_suzuki"  # one of: "perera_suzuki", "reizman_suzuki", "shields_direct_arylation", "baumgartner_cn_coupling"


In [ ]:
def load_perera_suzuki():
    df = pd.read_csv(DATA_DIR / "perera_suzuki.csv", comment="#")
    factors = {
        "electrophile": {"type": "categorical", "levels": sorted(df["electrophile"].unique().tolist())},
        "ligand":       {"type": "categorical", "levels": sorted(df["ligand"].unique().tolist())},
        "base":         {"type": "categorical", "levels": sorted(df["base"].unique().tolist())},
        "solvent":      {"type": "categorical", "levels": sorted(df["solvent"].unique().tolist())},
    }
    response_col = "yield"
    df = df.groupby(["electrophile", "ligand", "base", "solvent"], as_index=False)[response_col].mean()
    return df, factors, response_col


def load_reizman_suzuki():
    df = pd.read_csv(DATA_DIR / "reizman_suzuki.csv", comment="#")
    factors = {
        "catalyst":         {"type": "categorical", "levels": sorted(df["catalyst"].unique().tolist())},
        "t_res":            {"type": "continuous", "range": [float(df["t_res"].min()), float(df["t_res"].max())]},
        "temperature":      {"type": "continuous", "range": [float(df["temperature"].min()), float(df["temperature"].max())]},
        "catalyst_loading": {"type": "continuous", "range": [float(df["catalyst_loading"].min()), float(df["catalyst_loading"].max())]},
    }
    response_col = "yld"
    return df, factors, response_col


def load_shields_direct_arylation():
    df = pd.read_csv(DATA_DIR / "shields_direct_arylation.csv", comment="#")
    factors = {
        "Base_SMILES":    {"type": "categorical", "levels": sorted(df["Base_SMILES"].unique().tolist())},
        "Ligand_SMILES":  {"type": "categorical", "levels": sorted(df["Ligand_SMILES"].unique().tolist())},
        "Solvent_SMILES": {"type": "categorical", "levels": sorted(df["Solvent_SMILES"].unique().tolist())},
        "Concentration":  {"type": "continuous", "range": [float(df["Concentration"].min()), float(df["Concentration"].max())]},
        "Temp_C":         {"type": "continuous", "range": [float(df["Temp_C"].min()), float(df["Temp_C"].max())]},
    }
    response_col = "yield"
    return df, factors, response_col


def load_baumgartner_cn_coupling():
    df = pd.read_csv(DATA_DIR / "baumgartner_cn_coupling.csv", comment="#")
    factors = {
        "catalyst":         {"type": "categorical", "levels": sorted(df["catalyst"].unique().tolist())},
        "base":             {"type": "categorical", "levels": sorted(df["base"].unique().tolist())},
        "base_equivalents": {"type": "continuous", "range": [float(df["base_equivalents"].min()), float(df["base_equivalents"].max())]},
        "temperature":      {"type": "continuous", "range": [float(df["temperature"].min()), float(df["temperature"].max())]},
        "t_res":            {"type": "continuous", "range": [float(df["t_res"].min()), float(df["t_res"].max())]},
    }
    response_col = "yld"
    return df, factors, response_col


DATASET_LOADERS = {
    "perera_suzuki": load_perera_suzuki,
    "reizman_suzuki": load_reizman_suzuki,
    "shields_direct_arylation": load_shields_direct_arylation,
    "baumgartner_cn_coupling": load_baumgartner_cn_coupling,
}


def load_dataset(name):
    """Returns (df, factors_dict, response_col_normalized). `yield_norm` in [0,1] is added,
    normalized by dividing by this dataset's own max observed yield, so BO metrics (Section 8)
    are comparable in scale across datasets even though raw units/scales differ."""
    df, factors, response_col = DATASET_LOADERS[name]()
    df = df.copy()
    y = df[response_col].astype(float).clip(lower=0)
    df["yield_norm"] = (y / y.max()).clip(upper=1.0)
    return df, factors, "yield_norm"


df, FACTORS, RESPONSE_COL = load_dataset(DATASET)
print(f"Dataset: {DATASET}  |  {len(df)} rows  |  {len(FACTORS)} factors")
for name, spec in FACTORS.items():
    if spec["type"] == "categorical":
        print(f"  - {name}: categorical, {len(spec['levels'])} levels")
    else:
        print(f"  - {name}: continuous, range {spec['range']}")
df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.hist(df[RESPONSE_COL], bins=25, color="#4C72B0", edgecolor="white")
ax.set_xlabel("normalized yield (yield / max observed yield)")
ax.set_ylabel("count")
ax.set_title(f"{DATASET}: response distribution (n={len(df)})")
plt.tight_layout()
plt.show()

print(df[RESPONSE_COL].describe())


## 2 · Featurize the reaction

BO needs a numeric feature vector per condition. We support three interchangeable featurizations,
picked once via `FEATURIZATION` and **precomputed into a cached CSV** before any BO loop runs:

- `"onehot"` — one-hot encode every categorical factor, min-max scale continuous factors. Simple,
  no chemistry knowledge required, but treats e.g. every ligand as equally (dis)similar to every
  other ligand (no notion that two phosphines might be more alike than a phosphine and an amine).
- `"descriptors"` — physicochemical descriptors: for factors given as SMILES (e.g. Shields' base/
  ligand/solvent), RDKit descriptors (MolWt, TPSA, logP, num rotatable bonds, num H-bond donors/
  acceptors, ring count) computed directly from the SMILES; for factors given as bare categorical
  labels with no SMILES (e.g. Perera's `ligand_1_XPhos`-style names, Baumgartner's catalyst names),
  a small COSMO-RS-style placeholder descriptor set (steric bulk proxy, electronic-donating proxy,
  polarity proxy) assigned once per unique level -- in the spirit of Summit's COSMO-RS sigma-moment
  treatment described in `PAPERS-AND-DATASETS.md`, without requiring an actual COSMO-RS run. This
  lets a GP generalize across similar catalysts/ligands instead of treating each as an unrelated arm.
- `"random_projection"` — one-hot first, then a random linear projection down to a fixed dimension.
  A deliberately weak/naive baseline: it destroys the one-hot structure without adding chemical
  information, useful for showing students that *not every dimensionality-reduction trick helps.*

**Why precompute?** Every round of the BO loop needs to re-evaluate features for the *entire*
candidate pool (to pick the next batch) but the mapping condition→features never changes during a
run. Computing it once into `data/<dataset>_features_<featurization>.csv` means the BO loop itself
only ever does a fast pandas lookup — RDKit descriptor calculation in particular would otherwise be
the dominant cost of every acquisition step, for zero benefit (the features are static).


In [ ]:
FEATURIZATION = "onehot"  # one of: "onehot", "descriptors", "random_projection"
RANDOM_PROJECTION_DIM = 8  # only used if FEATURIZATION == "random_projection"


def _cosmo_like_placeholder(label, n_dims=3, seed_salt=0):
    """Deterministic pseudo-COSMO-RS descriptor for a bare categorical label with no SMILES:
    hashes the label string to 3 stable pseudo-physical numbers in [0, 1] (steric / electronic /
    polarity proxies). NOT a real COSMO-RS calculation -- a stand-in so categorical levels without
    structures still get a smooth, generalizable descriptor space instead of one-hot arms."""
    import hashlib
    out = []
    for i in range(n_dims):
        h = hashlib.md5(f"{label}_{i}_{seed_salt}".encode()).hexdigest()
        out.append((int(h[:8], 16) % 10_000) / 10_000.0)
    return np.array(out)


def _rdkit_descriptors_for_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return np.array([
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        Descriptors.NumRotatableBonds(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumHDonors(mol), Descriptors.RingCount(mol),
    ])


def featurize_onehot(df, factors):
    blocks = []
    for name, spec in factors.items():
        if spec["type"] == "categorical":
            dummies = pd.get_dummies(df[name], prefix=name)
            # ensure a stable column order/set across calls
            for lvl in spec["levels"]:
                col = f"{name}_{lvl}"
                if col not in dummies:
                    dummies[col] = 0
            dummies = dummies[[f"{name}_{lvl}" for lvl in spec["levels"]]]
            blocks.append(dummies.astype(float))
        else:
            scaler = MinMaxScaler()
            scaled = scaler.fit_transform(df[[name]].astype(float))
            blocks.append(pd.DataFrame(scaled, columns=[name], index=df.index))
    return pd.concat(blocks, axis=1).reset_index(drop=True)


def featurize_descriptors(df, factors):
    blocks = []
    for name, spec in factors.items():
        if spec["type"] == "categorical":
            looks_like_smiles = "SMILES" in name
            rows = []
            for lvl in df[name]:
                vec = None
                if looks_like_smiles:
                    vec = _rdkit_descriptors_for_smiles(lvl)
                if vec is None:
                    vec = _cosmo_like_placeholder(lvl)
                rows.append(vec)
            arr = np.vstack(rows)
            cols = [f"{name}_desc{i}" for i in range(arr.shape[1])]
            block = pd.DataFrame(arr, columns=cols, index=df.index)
            # rescale each descriptor column to [0, 1] so no single physical unit dominates the GP kernel
            block = pd.DataFrame(MinMaxScaler().fit_transform(block), columns=cols, index=df.index)
            blocks.append(block)
        else:
            scaler = MinMaxScaler()
            scaled = scaler.fit_transform(df[[name]].astype(float))
            blocks.append(pd.DataFrame(scaled, columns=[name], index=df.index))
    return pd.concat(blocks, axis=1).reset_index(drop=True)


def featurize_random_projection(df, factors, out_dim=RANDOM_PROJECTION_DIM, seed=0):
    onehot = featurize_onehot(df, factors)
    proj = GaussianRandomProjection(n_components=min(out_dim, onehot.shape[1]), random_state=seed)
    projected = proj.fit_transform(onehot.values)
    cols = [f"rp_{i}" for i in range(projected.shape[1])]
    return pd.DataFrame(projected, columns=cols, index=onehot.index)


FEATURIZERS = {
    "onehot": featurize_onehot,
    "descriptors": featurize_descriptors,
    "random_projection": featurize_random_projection,
}


def get_features(dataset_name, df, factors, featurization):
    """Computes (or loads a cached copy of) the feature matrix for the FULL candidate pool once.
    The BO loop below only ever indexes into this cached matrix -- no recomputation at query time."""
    cache_path = DATA_DIR / f"{dataset_name}_features_{featurization}.csv"
    if cache_path.exists():
        return pd.read_csv(cache_path)
    feats = FEATURIZERS[featurization](df, factors)
    feats.to_csv(cache_path, index=False)
    return feats


X_df = get_features(DATASET, df, FACTORS, FEATURIZATION)
X = X_df.values.astype(float)
y = df[RESPONSE_COL].values.astype(float)
print(f"Featurization '{FEATURIZATION}' -> feature matrix shape {X.shape}")
X_df.head()


## 3 · Choose an initialization technique

Before any GP can be fit, we need a handful of starting observations. `N_INIT = 10` is fixed so
different choices below stay comparable.

- `"random"` — uniformly sample `N_INIT` rows from the candidate pool.
- `"kmeans"` — cluster the featurized candidate pool into `N_INIT` clusters and take the point
  nearest each centroid. This spreads the initial batch out to cover the condition space more
  evenly than random sampling would (useful when the pool is small or unevenly distributed), at
  the cost of losing pure randomness (which matters if you want unbiased seed-to-seed variance --
  see Section 8).


In [ ]:
INIT_METHOD = "random"  # one of: "random", "kmeans"
N_INIT = 10


def init_random(X, n_init, rng):
    idx = rng.choice(len(X), size=n_init, replace=False)
    return idx


def init_kmeans(X, n_init, rng):
    km = KMeans(n_clusters=n_init, n_init=10, random_state=int(rng.integers(0, 1_000_000)))
    labels = km.fit_predict(X)
    idx = []
    for k in range(n_init):
        cluster_pts = np.where(labels == k)[0]
        if len(cluster_pts) == 0:
            continue
        centroid = km.cluster_centers_[k]
        dists = np.linalg.norm(X[cluster_pts] - centroid, axis=1)
        idx.append(cluster_pts[np.argmin(dists)])
    return np.array(idx)


INIT_METHODS = {"random": init_random, "kmeans": init_kmeans}

_rng_preview = np.random.default_rng(GLOBAL_SEED)
init_idx_preview = INIT_METHODS[INIT_METHOD](X, N_INIT, _rng_preview)
print(f"Init method '{INIT_METHOD}' selected {len(init_idx_preview)} starting points; "
      f"best initial normalized yield = {y[init_idx_preview].max():.3f}")


## 4 · Choose an acquisition function and its hyperparameters

Given the GP surrogate's posterior mean $\mu(x)$ and uncertainty $\sigma(x)$ at candidate $x$, the
acquisition function scores how worthwhile it is to query $x$ next:

- **EI (Expected Improvement)** — expected amount by which $x$ beats the current best observation
  $y^*$; automatically balances exploration (high $\sigma$) and exploitation (high $\mu$) with no
  hyperparameter to tune.
- **UCB (Upper Confidence Bound)** — $\mu(x) + \sqrt{\beta}\,\sigma(x)$; a single, user-editable
  knob `beta` directly controls the explore/exploit trade-off (higher `beta` -> more exploration).
- **qEI (batch Expected Improvement)** — the batch-aware generalization of EI used when proposing
  more than one point per round (needed here since `BATCH_SIZE = 4`, Section 5); jointly scores a
  set of `q` candidates via Monte Carlo integration over the GP posterior instead of picking the
  top-`q` single-point EI points independently (which would tend to cluster all `q` points in the
  same high-EI region).


In [ ]:
ACQUISITION = "EI"  # one of: "EI", "UCB", "qEI"
ACQUISITION_HPARAMS = {"beta": 2.0}  # only used by UCB


def make_acqf(model, best_f, acquisition, hparams):
    if acquisition == "EI":
        return ExpectedImprovement(model=model, best_f=best_f)
    elif acquisition == "UCB":
        return UpperConfidenceBound(model=model, beta=hparams.get("beta", 2.0))
    elif acquisition == "qEI":
        return qExpectedImprovement(model=model, best_f=best_f)
    else:
        raise ValueError(f"Unknown acquisition: {acquisition}")


print(f"Acquisition: {ACQUISITION}  hparams: {ACQUISITION_HPARAMS}")


## 5 · Batch size

`BATCH_SIZE = 4` is **fixed, not user-editable** — every automated/flow platform in the source
papers runs several reactions in parallel per round (Perera's platform ran continuously, Reizman/
Baumgartner's flow rigs and Shields' physical robotic platform all dispatch a handful of reactions
concurrently before waiting on the next round of results), so a batch of `q=4` per round is the
realistic unit of "one round" here. Keeping it fixed across all submissions also means the
leaderboard (Section 8) is comparing apples to apples — everyone is optimizing under the same
per-round experimental throughput, so differences in `auc_mean` reflect featurization/init/
acquisition choices, not who gave themselves a bigger batch.


In [ ]:
BATCH_SIZE = 4  # fixed -- see markdown above; do not change

def optimize_batch(model, best_f, acquisition, hparams, bounds, q=BATCH_SIZE):
    """Returns q new candidate points (in normalized [0,1]^d space) by optimizing the chosen
    acquisition function with BoTorch's optimize_acqf, using its q-batch support."""
    if acquisition in ("EI", "UCB"):
        # single-point acquisition functions: BoTorch still supports q>1 via sequential greedy
        # optimization (each of the q points optimized in turn, conditioning on the previous picks)
        acqf = make_acqf(model, best_f, acquisition, hparams)
        candidates, _ = optimize_acqf(
            acq_function=acqf, bounds=bounds, q=q, num_restarts=8, raw_samples=128,
            sequential=True,
        )
    else:  # qEI: natively batch-aware, jointly optimizes all q points via MC integration
        acqf = make_acqf(model, best_f, acquisition, hparams)
        candidates, _ = optimize_acqf(
            acq_function=acqf, bounds=bounds, q=q, num_restarts=8, raw_samples=128,
        )
    return candidates


## 6 · Preview the trajectory on a 2D plot before committing

Before launching the expensive 5-seed run (Section 8), run **one quick preview**: a single seed, a
small number of rounds, using exactly the config chosen in Sections 1-5. We project the full
featurized candidate pool to 2D with PCA (cheap, deterministic, no extra dependency beyond
scikit-learn) and plot:

- all candidate points in light grey (the full condition space),
- the queried points colored by round order (a sequential colormap — darker/lighter = earlier
  round), with marker size or a second visual channel encoding the observed yield,
- the best point found, annotated.

**What to look for.** Are later rounds clustering near the best-yield region, or still scattered
across the space? If points are still scattered even in the last round, that is a sign your
featurization may not capture what actually drives yield (try `"descriptors"` instead of
`"onehot"`), or your acquisition function is over-exploring (lower UCB's `beta`, or try EI) —
better to catch and fix this now than after spending the full 5-seed x 8-round budget in Section 8.


In [ ]:
def run_bo_loop(dataset_name, featurization, init_method, acquisition, acquisition_hparams,
                 batch_size, n_rounds, seed, X=None, y=None, X_df=None):
    """Closed-loop BO simulation against the fixed lookup-table dataset. Returns a dict with the
    per-round trajectory (best-so-far normalized yield) and the full query history."""
    if X is None or y is None:
        _df, _factors, _resp = load_dataset(dataset_name)
        _X_df = get_features(dataset_name, _df, _factors, featurization)
        X, y = _X_df.values.astype(float), _df[_resp].values.astype(float)

    rng = np.random.default_rng(seed)
    bounds_np = np.stack([X.min(axis=0), X.max(axis=0)])
    # guard against zero-width dims (a constant feature column) which would break normalization
    span = bounds_np[1] - bounds_np[0]
    span[span == 0] = 1.0
    bounds = torch.tensor(bounds_np, dtype=torch.float64)

    init_idx = INIT_METHODS[init_method](X, N_INIT, rng)
    queried_idx = list(init_idx)
    round_of_query = [0] * len(init_idx)

    best_so_far = [float(y[queried_idx].max())]

    for r in range(1, n_rounds + 1):
        train_X = torch.tensor(X[queried_idx], dtype=torch.float64)
        train_Y = torch.tensor(y[queried_idx], dtype=torch.float64).unsqueeze(-1)

        gp = SingleTaskGP(train_X, train_Y, covar_module=ScaleKernel(MaternKernel(nu=2.5, ard_num_dims=X.shape[1])))
        mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
        fit_gpytorch_mll(mll)

        best_f = train_Y.max().item()
        candidates = optimize_batch(gp, best_f, acquisition, acquisition_hparams, bounds, q=batch_size)
        candidates_np = candidates.detach().numpy()

        # "query" = nearest-neighbor lookup in the fixed dataset (no real experiment is run)
        for cand in candidates_np:
            dists = np.linalg.norm(X - cand, axis=1)
            nearest = int(np.argmin(dists))
            queried_idx.append(nearest)
            round_of_query.append(r)

        best_so_far.append(float(y[queried_idx].max()))

    return {
        "queried_idx": np.array(queried_idx),
        "round_of_query": np.array(round_of_query),
        "best_so_far": np.array(best_so_far),  # length n_rounds + 1 (index 0 = after init)
        "X": X, "y": y,
    }


# --- quick preview: 1 seed, 4 rounds of batch 4 = 16 queries beyond the 10 init points ---
PREVIEW_N_ROUNDS = 4
preview = run_bo_loop(DATASET, FEATURIZATION, INIT_METHOD, ACQUISITION, ACQUISITION_HPARAMS,
                       BATCH_SIZE, PREVIEW_N_ROUNDS, seed=GLOBAL_SEED, X=X, y=y, X_df=X_df)
print("Preview best-so-far normalized yield by round:", np.round(preview["best_so_far"], 3))


In [ ]:
pca = PCA(n_components=2, random_state=0)
X_2d = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X_2d[:, 0], X_2d[:, 1], color="lightgrey", s=12, label="full candidate pool", zorder=1)

qi, ro = preview["queried_idx"], preview["round_of_query"]
cmap = cm.get_cmap("viridis")
n_rounds_total = ro.max()
sizes = 30 + 200 * (y[qi] - y[qi].min()) / max(y[qi].max() - y[qi].min(), 1e-9)
sc = ax.scatter(X_2d[qi, 0], X_2d[qi, 1], c=ro, cmap=cmap, s=sizes,
                 edgecolor="black", linewidth=0.4, zorder=2)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("round (0 = init)")

best_pos = qi[np.argmax(y[qi])]
ax.scatter(*X_2d[best_pos], marker="*", s=400, color="red", edgecolor="black", zorder=3,
           label=f"best found (yield={y[best_pos]:.2f})")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title(f"{DATASET} | {FEATURIZATION} | {INIT_METHOD} init | {ACQUISITION} preview trajectory")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()


**Sanity check before committing:** does the preview trajectory above show later rounds
(yellow/bright points) concentrating near the best-yield region (marked with a red star), or still
scattered across the grey cloud like the early rounds (dark purple points)? If it looks scattered
even by the last round, reconsider your featurization or acquisition choice in Sections 2 and 4
*before* running Section 8's full 5-seed job.


## 7 · Submit the design choices

A checkpoint before the expensive part: assemble every design decision made in Sections 1-5 into one
`design` dict, print it to confirm, and write it to a local JSON file.


In [ ]:
N_ROUNDS = 8  # fixed for the full run in Section 8 -- see markdown there

design = {
    "dataset": DATASET,
    "featurization": FEATURIZATION,
    "init_method": INIT_METHOD,
    "acquisition": ACQUISITION,
    "acquisition_hparams": ACQUISITION_HPARAMS,
    "batch_size": BATCH_SIZE,
    "n_rounds": N_ROUNDS,
}
print(json.dumps(design, indent=2))

with open("design_choices.json", "w") as f:
    json.dump(design, f, indent=2)
print("Saved design_choices.json")


## 8 · Launch the run for 5 seeds, submit to a leaderboard

Now run the **full** closed-loop BO simulation for `seed in [0, 1, 2, 3, 4]`, each seed doing
`N_INIT=10` initialization + `N_ROUNDS=8` rounds of batch-`BATCH_SIZE=4` acquisition (32 queries
beyond init, 42 total per seed) against the fixed lookup-table dataset chosen in Section 1 — as in
Section 6, this is a closed-loop simulation against a fixed historical dataset, standard practice
per Perera/Reizman/Summit's own benchmark-emulator methodology, **not** new wet-lab experiments.

For each seed we track the normalized best-so-far yield after every round. From the 5 resulting
curves we compute:

- **AUC** — trapezoidal area under each seed's best-so-far-vs-round curve, normalized by `n_rounds`
  to `[0, 1]` (a curve that instantly finds the max scores ~1; one that never improves past its
  init score scores close to its init value), averaged over the 5 seeds.
- **std** — standard deviation of those 5 per-seed AUC values (lower = more consistent/reliable
  across random seeds — recall Shields et al.'s finding that BO's main advantage over human experts
  was *consistency*, not just best-case performance).
- **max score achieved** — the maximum normalized best-so-far yield reached by any seed by the
  final round.


In [ ]:
SEEDS = [0, 1, 2, 3, 4]

def best_so_far_auc(best_so_far, n_rounds):
    """Trapezoidal AUC of the best-so-far-vs-round curve, normalized to [0, 1] by n_rounds
    (x-axis span) -- NOT by the max possible y value, so a curve that quickly saturates near 1.0
    scores close to 1.0, while one that stays flat near its init value scores close to that value."""
    x = np.arange(len(best_so_far))  # 0..n_rounds
    auc = np.trapz(best_so_far, x)
    return auc / n_rounds


all_curves = []
per_seed_auc = []
t0 = time.time()
for seed in SEEDS:
    result = run_bo_loop(design["dataset"], design["featurization"], design["init_method"],
                          design["acquisition"], design["acquisition_hparams"],
                          design["batch_size"], design["n_rounds"], seed=seed, X=X, y=y, X_df=X_df)
    all_curves.append(result["best_so_far"])
    per_seed_auc.append(best_so_far_auc(result["best_so_far"], design["n_rounds"]))
    print(f"seed {seed}: best-so-far final = {result['best_so_far'][-1]:.3f}  AUC = {per_seed_auc[-1]:.3f}"
          f"  ({time.time()-t0:.0f}s elapsed)")

all_curves = np.stack(all_curves)  # shape (n_seeds, n_rounds+1)
auc_mean = float(np.mean(per_seed_auc))
auc_std = float(np.std(per_seed_auc))
max_score_achieved = float(all_curves[:, -1].max())

print(f"\nauc_mean = {auc_mean:.4f}   auc_std = {auc_std:.4f}   max_score_achieved = {max_score_achieved:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
rounds = np.arange(all_curves.shape[1])
for i, seed in enumerate(SEEDS):
    ax.plot(rounds, all_curves[i], color="grey", alpha=0.4, linewidth=1)

mean_curve = all_curves.mean(axis=0)
std_curve = all_curves.std(axis=0)
ax.plot(rounds, mean_curve, color="#C44E52", linewidth=2.5, label="mean best-so-far")
ax.fill_between(rounds, mean_curve - std_curve, mean_curve + std_curve, color="#C44E52", alpha=0.2, label="+/- 1 std")

ax.set_xlabel("round (0 = after init)")
ax.set_ylabel("best-so-far normalized yield")
ax.set_title(f"{design['dataset']} | {design['featurization']} | {design['init_method']} | "
             f"{design['acquisition']} | {len(SEEDS)} seeds")
ax.legend()
plt.tight_layout()
plt.show()


### Submit to the leaderboard

Ranking on the leaderboard is done **per dataset** (a run on `perera_suzuki` is not directly
comparable to one on `baumgartner_cn_coupling`), sorted by `auc_mean` descending within each dataset.
The submission is a **self-report**: you compute `auc_mean`/`auc_std`/`max_score_achieved` locally
(above) and POST them — there is no independent instructor grading/oracle for this tutorial (unlike
`08-molecule-generation`'s docking oracle), so the honor system applies. This mirrors the
`06-pretraining_finetuning` leaderboard pattern exactly (`validate_payload` / `write_payload_json` /
`append_payload_csv` / `submit_payload`, adapted field list) — see
`leaderboard/README.md` and `leaderboard/SETUP.md` in this folder for the full Google Sheets +
Apps Script setup this posts to.


In [ ]:
import sys
sys.path.insert(0, "leaderboard")
from submit_payload import validate_payload, write_payload_json, append_payload_csv, submit_payload, utc_now_iso

# ---- Fill this in once per team; keep it stable across resubmissions so the leaderboard can
# ---- track your best run per dataset. ----
TEAM_NAME = "team-example"
LEADERBOARD_ENDPOINT_URL = "https://script.google.com/a/macros/caltech.edu/s/AKfycbxoFbsNPkJ5ADbwF7kt_Obz7e91luxWN9TECn0AdtQ45-ukct2PjUEHqgnH3wrwQXfw/exec"
# ---------------------------------------------------------------------------------------------

payload = {
    "run_id": str(uuid.uuid4()),
    "timestamp_utc": utc_now_iso(),
    "team_name": TEAM_NAME,
    "dataset": design["dataset"],
    "featurization": design["featurization"],
    "init_method": design["init_method"],
    "acquisition": design["acquisition"],
    "acquisition_hparams": json.dumps(design["acquisition_hparams"]),
    "batch_size": design["batch_size"],
    "n_rounds": design["n_rounds"],
    "n_seeds": len(SEEDS),
    "auc_mean": round(auc_mean, 4),
    "auc_std": round(auc_std, 4),
    "max_score_achieved": round(max_score_achieved, 4),
    "notebook_version": "reaction-bo.ipynb v1",
    "notes": "",
}
print(json.dumps(payload, indent=2))

write_payload_json(payload, "leaderboard_submission.json")
append_payload_csv(payload, "local_leaderboard_log.csv")

if LEADERBOARD_ENDPOINT_URL.startswith("PASTE_"):
    print("\nLEADERBOARD_ENDPOINT_URL not set -- payload saved locally only. "
          "See leaderboard/SETUP.md to get the real endpoint from your instructor.")
else:
    response = submit_payload(payload, LEADERBOARD_ENDPOINT_URL)
    print("Leaderboard response:", response)


## Exercises

**Exercise 1 (easy-medium).** Re-run Sections 1-8 with `FEATURIZATION = "descriptors"` instead of
`"onehot"` (same dataset, init method, acquisition). Compare `auc_mean` and `auc_std` between the
two runs. Does giving the GP physically-meaningful descriptors instead of bare one-hot indicators
change how quickly/reliably it finds high-yield conditions? Why might this matter more for
`shields_direct_arylation` (SMILES available) than for `baumgartner_cn_coupling` (no SMILES,
placeholder descriptors only)?


In [ ]:
### YOUR CODE HERE ###
# Re-run the Section 1-8 pipeline with FEATURIZATION = "descriptors" and compare auc_mean/auc_std
# against your onehot run above. You can call run_bo_loop(...) directly across SEEDS again, or
# re-execute the notebook top-to-bottom after changing the FEATURIZATION cell.


**Exercise 2 (medium).** Implement one more acquisition function: **Probability of Improvement
(PI)**, $\mathrm{PI}(x) = \Phi\!\left(\frac{\mu(x) - y^* - \xi}{\sigma(x)}\right)$ for some
small exploration margin $\xi$ (try $\xi=0.01$). BoTorch provides
`botorch.acquisition.ProbabilityOfImprovement` — add it as a fourth option in `make_acqf` (Section
4) and `optimize_batch` (Section 5), then compare its preview trajectory (Section 6) against EI's.


In [ ]:
### YOUR CODE HERE ###
# from botorch.acquisition import ProbabilityOfImprovement
# Add "PI" as a branch in make_acqf() and optimize_batch() above, then re-run the Section 6 preview
# with ACQUISITION = "PI" and compare its 2D trajectory plot against the EI preview.


**Exercise 3 (hard, optional).** The `"kmeans"` initializer (Section 3) spends its whole budget
on spreading points evenly across the *feature* space, which may not correlate well with spreading
across *yield* — two nearby conditions in feature space can still have very different yields.
Implement a third init method, `"kmeans_then_best_per_cluster"`: cluster as in `"kmeans"`, but from
each cluster pick the *highest-yield* point instead of the point nearest the centroid. This is a
form of "cheating" (using the label to pick the init set) — discuss in a markdown cell why this
would NOT be a fair option to leave enabled for the Section 8 leaderboard run, even though it's a
handy debugging tool while developing the notebook.


In [ ]:
### YOUR CODE HERE ###
# def init_kmeans_then_best_per_cluster(X, n_init, rng, y=None):
#     ...
# Discuss (markdown, not code): why must the real Section 8 run only use label-blind init methods?
